In [1]:
import pandas as pd

crop = pd.read_csv("../data/raw/crop_production.csv")
weather = pd.read_csv("../data/raw/GlobalLandTemperaturesByCountry.csv")
logistics = pd.read_csv("../data/raw/logistics.csv")


In [2]:
# 1. Handle missing production values
crop["Production"] = crop["Production"].fillna(crop["Production"].median())

# 2. Remove impossible values
crop = crop[crop["Area"] > 0]

# 3. Create yield feature
crop["Yield"] = crop["Production"] / crop["Area"]

crop.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,Yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


In [3]:
weather_india = weather[weather["Country"] == "India"]
weather_india = weather_india[["dt", "AverageTemperature"]]


In [4]:
weather_india["AverageTemperature"] = weather_india["AverageTemperature"].interpolate()
weather_india["dt"] = pd.to_datetime(weather_india["dt"])

weather_india.head()


,dt,AverageTemperature
243695,1796-01-01,17.044
243696,1796-02-01,19.193
243697,1796-03-01,22.319
243698,1796-04-01,27.233
243699,1796-05-01,30.035


In [5]:
# 1. Convert timestamps to datetime (safe way)
logistics["delivery_time_hours"] = pd.to_datetime(
    logistics["delivery_time_hours"], errors="coerce"
)

logistics["expected_time_hours"] = pd.to_datetime(
    logistics["expected_time_hours"], errors="coerce"
)

# 2. Convert datetime to numeric hours
logistics["delivery_time_hours"] = (
    logistics["delivery_time_hours"].dt.hour +
    logistics["delivery_time_hours"].dt.minute / 60 +
    logistics["delivery_time_hours"].dt.second / 3600
)

logistics["expected_time_hours"] = (
    logistics["expected_time_hours"].dt.hour +
    logistics["expected_time_hours"].dt.minute / 60 +
    logistics["expected_time_hours"].dt.second / 3600
)

# 3. Create Delivery_Delay
logistics["Delivery_Delay"] = (
    logistics["delivery_time_hours"] - logistics["expected_time_hours"]
)

# 4. Replace negative delays with 0
logistics["Delivery_Delay"] = logistics["Delivery_Delay"].clip(lower=0)

# 5. Convert delayed yes/no to binary
logistics["Delayed_Binary"] = logistics["delayed"].map({"yes": 1, "no": 0})

logistics[["Delivery_Delay", "Delayed_Binary"]].head()


,Delivery_Delay,Delayed_Binary
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0


In [6]:
crop["Yield"] = crop["Production"] / crop["Area"]

In [7]:
crop["Yield"] = crop["Yield"].fillna(crop["Yield"].median())

In [8]:
def assign_risk(row):
    if row["Yield"] < crop["Yield"].quantile(0.25):
        return "High"
    elif row["Yield"] < crop["Yield"].quantile(0.50):
        return "Medium"
    else:
        return "Low"

crop["Waste_Risk"] = crop.apply(assign_risk, axis=1)

crop["Waste_Risk"].value_counts()

Waste_Risk
Low       128134
High       61523
Medium     56434
Name: count, dtype: int64

In [9]:
crop_small = crop.sample(50000, random_state=42)


In [10]:
# Add year column to weather
weather_india["Year"] = weather_india["dt"].dt.year

# Merge crop + weather on year
merged = crop_small.merge(
    weather_india.groupby("Year")["AverageTemperature"].mean().reset_index(),
    left_on="Crop_Year",
    right_on="Year",
    how="left"
)

merged.head()


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,Yield,Waste_Risk,Year,AverageTemperature
0,Karnataka,UDUPI,2005,Rabi,Horse-gram,1122.0,836.0,0.745098,Medium,2005.0,24.495417
1,Madhya Pradesh,GWALIOR,2003,Whole Year,Brinjal,194.0,0.0,0.000000,High,2003.0,24.649917
2,Andhra Pradesh,CHITTOOR,2010,Rabi,Sesamum,334.0,118.0,0.353293,High,2010.0,25.050833
3,Andhra Pradesh,KRISHNA,2014,Rabi,Tomato,538.0,7289.0,13.548327,Low,NaN,NaN
4,Uttar Pradesh,SULTANPUR,2011,Rabi,Coriander,59.0,33.0,0.559322,Medium,2011.0,24.415583


In [11]:
merged.to_csv("../data/processed/final_dataset.csv", index=False)

In [13]:
df = merged.copy()

In [14]:
print(df.columns)

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production', 'Yield', 'Waste_Risk', 'Year', 'AverageTemperature'],
      dtype='object')


In [27]:
df = df.drop(columns=["Yield", "Production"], errors="ignore")

print("Remaining columns:")
print(df.columns)

Remaining columns:
Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'waste_risk', 'Year', 'AverageTemperature'],
      dtype='object')


In [28]:
df = df.rename(columns={"Waste_Risk": "waste_risk"})

y = df["waste_risk"]
X = df.drop("waste_risk", axis=1)

In [29]:

print(y.value_counts())

waste_risk
Low       26132
High      12500
Medium    11368
Name: count, dtype: int64


In [30]:
X = X.drop(columns=["District_Name"])

In [31]:
categorical_cols = X.select_dtypes(include=["object"]).columns
print(categorical_cols)

Index(['State_Name', 'Season', 'Crop'], dtype='object')


In [32]:
for col in categorical_cols:
    print(col, X[col].nunique())

State_Name 33
Season 6
Crop 122


In [34]:
X = pd.get_dummies(X, drop_first=True)

In [35]:
print(X.shape)
print(y.shape)

(50000, 162)
(50000,)


In [36]:
X_final = X.copy()
y_final = y.copy()

X_final.to_csv("../data/processed/X_final.csv", index=False)
y_final.to_csv("../data/processed/y_final.csv", index=False)

print("Clean dataset saved.")

Clean dataset saved.


In [37]:
import os
print(os.listdir("../data/processed"))

['final_dataset.csv', 'X_final.csv', 'y_final.csv']
